# 05 - Intervention Modeling & Recovery Simulation

**Modeling Admissible Treatments for Trisomy 21 using CGCS**  
**Allen Lab Collaboration**

In [ ]:
# ================================================
# SETUP - Run this first
# ================================================
import sys
from pathlib import Path

if "/content/allen-lab-report-tool" not in str(Path.cwd()):
    repo_path = "/content/allen-lab-report-tool"
    if Path(repo_path).exists():
        %cd /content/allen-lab-report-tool
    else:
        print("Cloning repo...")
        !git clone https://github.com/thinkthoughts/allen-lab-report-tool.git
        %cd allen-lab-report-tool

sys.path.insert(0, str(Path.cwd() / "src"))

import grok
from grok.trisomy_metrics import trisomy_cgcs_score, simulate_intervention_recovery
from grok.visualization import plot_cgcs_vs_noise

import numpy as np
import matplotlib.pyplot as plt

print("✅ Intervention Modeling Notebook Ready")
print(f"Version: {grok.__version__}")

## 1. Baseline Trisomy 21 CGCS

In [ ]:
baseline = trisomy_cgcs_score(
    dosage_ratio=1.5,
    overexpression_imbalance=0.5,
    global_dysregulation=0.4,
    return_components=True
)

print("Baseline Trisomy 21 CGCS:")
for k, v in baseline.items():
    print(f"  {k:25} = {v:.4f}")

## 2. Modeling Different Intervention Types

In [ ]:
interventions = {
    "Early Therapies (PT/OT/Speech)": 0.35,
    "Targeted Pharmacological (e.g. DYRK1A inhibitor)": 0.45,
    "Combined Approach": 0.60,
    "Strong Hypothetical": 0.75
}

print("Intervention Recovery Simulation:\n")
results = {}
for name, strength in interventions.items():
    recovered = simulate_intervention_recovery(baseline['cgcs'], strength)
    results[name] = recovered
    print(f"{name:35} → CGCS = {recovered:.4f}  (+{recovered - baseline['cgcs']:.4f})")

## 3. Recovery Curve Visualization

In [ ]:
strengths = np.linspace(0.0, 0.8, 20)
recovered_scores = [simulate_intervention_recovery(baseline['cgcs'], s) for s in strengths]

plt.figure(figsize=(9, 6))
plt.plot(strengths*100, recovered_scores, 'o-', color='green', linewidth=2.5, label='Recovered CGCS')
plt.axhline(y=baseline['cgcs'], color='red', linestyle='--', label='Baseline Trisomy 21')
plt.axhline(y=0.85, color='blue', linestyle=':', label='High Coherence Target')

plt.title('CGCS Recovery as Function of Intervention Strength')
plt.xlabel('Intervention Effectiveness (%)')
plt.ylabel('Achieved CGCS')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Summary Table

In [ ]:
import pandas as pd

data = {
    "Intervention Type": list(interventions.keys()),
    "Strength": list(interventions.values()),
    "Baseline CGCS": [baseline['cgcs']] * len(interventions),
    "Recovered CGCS": list(results.values()),
    "Improvement": [results[k] - baseline['cgcs'] for k in interventions.keys()]
}

df = pd.DataFrame(data)
display(df.round(4))

---
**Key Insight**:  
Even moderate admissible interventions (early therapies + targeted drugs) can meaningfully improve effective coherence (CGCS) in the model. This framework allows quantitative comparison of different treatment strategies.